# Đề 6 — Classification: Book Recommendation Dataset

## Định nghĩa bài toán

Dự đoán một lượt đánh giá sách thuộc nhóm điểm cao (8–10)
hay nhóm chưa đạt mức điểm cao (1–7), dựa trên thông tin
người dùng và sách.

Chỉ xét các lượt đánh giá có điểm từ 1–10. Nhóm quy ước:
- `liked = 1`: điểm 8–10 — nhóm đánh giá cao.
- `liked = 0`: điểm 1–7 — nhóm chưa đạt mức đánh giá cao.

Ngưỡng 8 do nhóm lựa chọn để phân biệt các đánh giá ở vùng
điểm cao của thang 1–10. Đây là quy ước của bài toán,
không phải ngưỡng được đề bài quy định.

Điểm 0 không được gán vào hai nhãn trên và được loại khỏi
tập dữ liệu phân loại.

In [ ]:
# Nhập thư viện: Path xử lý đường dẫn; pandas xử lý bảng; numpy hỗ trợ tính toán; display hiển thị bảng.
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from scipy.stats import mannwhitneyu
from IPython.display import display
# Chọn thư mục chứa dữ liệu CSV; kiểm tra thư mục tồn tại trước khi đọc.
DATA_DIR = Path("DATA_SET")
assert DATA_DIR.is_dir(), "Cần đặt thư mục DATA_SET trong thư mục làm việc hoặc sửa DATA_DIR."


: 

## 1. Data Inspection
Đọc tự động ba bảng CSV. Kiểm tra UTF-8 bằng chế độ strict trên toàn bộ tệp; giữ ISBN dạng chuỗi để bảo toàn số 0 đầu và ký tự X. Ba bảng khác cấu trúc nên dùng merge thay vì concat.

In [ ]:
# Tạo từ điển lưu ba bảng dữ liệu theo tên Books, Users và Ratings.
tables = {}

# Lần lượt đọc từng tệp CSV trong thư mục dữ liệu.
for name in ["Books", "Users", "Ratings"]:
    path = DATA_DIR / f"{name}.csv"
    # Đọc toàn bộ tệp với UTF-8 strict để phát hiện lỗi mã hóa; pass chỉ bỏ qua nội dung từng dòng.
    with path.open(encoding="utf-8", errors="strict") as stream:
        for _ in stream:
            pass
    # Nạp CSV vào DataFrame; giữ ISBN dạng chuỗi để không mất số 0 đầu hoặc ký tự X.
    tables[name] = pd.read_csv(path, encoding="utf-8", dtype={"ISBN": "string"}, low_memory=False)

# Lấy ba DataFrame từ từ điển để dùng ở các bước sau.
books, users, ratings = [tables[n] for n in ["Books", "Users", "Ratings"]]

# Tóm tắt mỗi bảng: shape[0] là số dòng, shape[1] là số cột; duplicated đánh dấu dòng trùng.
summary = pd.DataFrame([{"Bảng": n, "Số dòng": d.shape[0], "Số cột": d.shape[1], "Dòng trùng": int(d.duplicated().sum())} for n,d in tables.items()])
display(summary)

# Hiển thị kiểu dữ liệu và số ô thiếu của từng cột trong mỗi bảng.
for name, frame in tables.items():
    print("\n", name)
    # dtypes lấy kiểu từng cột; astype(str) chuyển thông tin kiểu thành chuỗi, không đổi dữ liệu gốc.
    display(pd.DataFrame({"Kiểu": frame.dtypes.astype(str), "Thiếu": frame.isna().sum()}))


In [ ]:
# Kiểm tra khóa ISBN và User-ID không bị thiếu và không trùng; assert báo lỗi nếu điều kiện sai.
assert books["ISBN"].notna().all() and books["ISBN"].is_unique
assert users["User-ID"].notna().all() and users["User-ID"].is_unique

# Đếm các dòng đánh giá trùng cặp người dùng–sách.
print("Trùng cặp User-ID, ISBN:", ratings.duplicated(["User-ID", "ISBN"]).sum())

# isin kiểm tra ISBN có trong bảng sách; ~ đảo kết quả; sum đếm các đánh giá không tìm thấy sách.
print("Đánh giá thiếu sách:", (~ratings["ISBN"].isin(books["ISBN"])).sum())

# Đếm các đánh giá có User-ID không xuất hiện trong bảng người dùng.
print("Đánh giá thiếu người dùng:", (~ratings["User-ID"].isin(users["User-ID"])).sum())

# Đếm số lượt cho từng mức điểm.
display(ratings.groupby("Book-Rating").size().rename("Số lượt").to_frame())

# Chuyển năm xuất bản sang số; errors="coerce" biến giá trị không chuyển được thành NaN.
year = pd.to_numeric(books["Year-Of-Publication"], errors="coerce")

# loc chọn những dòng có năm thiếu hoặc không chuyển được thành số và các cột cần xem xét.
display(books.loc[year.isna(), ["ISBN", "Book-Title", "Book-Author", "Year-Of-Publication", "Publisher"]])

# Kiểm tra tuổi nhỏ nhất và lớn nhất để phát hiện giá trị bất thường, ví dụ 0 hoặc 244.
print("Tuổi nhỏ nhất/lớn nhất:", users["Age"].min(), users["Age"].max())


## 2. Data Loading and Integration
Ratings là bảng trung tâm, mỗi dòng là một lượt đánh giá. Ratings có quan hệ nhiều–một với Users qua cột User-ID và với Books qua cột ISBN: một người dùng có thể có nhiều lượt đánh giá, một sách có thể được đánh giá nhiều lần. Dùng left join để giữ toàn bộ lượt đánh giá, kể cả khi không tìm thấy người dùng hoặc sách tương ứng. Tham số `validate='many_to_one'` kiểm tra khóa ở bảng Users hoặc Books không bị trùng; nếu bị trùng thì báo lỗi để tránh ghép làm tăng số dòng. Chỉ ghép các cột cần phân tích để giảm bộ nhớ.

In [ ]:
# Chọn các cột thông tin sách cần ghép để giảm bộ nhớ.
book_columns = ["ISBN", "Book-Title", "Book-Author", "Year-Of-Publication", "Publisher"]

# Ghép đánh giá với sách theo ISBN; left giữ mọi đánh giá, Tham số `validate='many_to_one'` kiểm tra khóa ở bảng Users hoặc Books không bị trùng; nếu bị trùng thì báo lỗi để tránh ghép làm tăng số dòng. Chỉ ghép các cột cần phân tích để giảm bộ nhớ.; indicator ghi trạng thái khớp của mỗi lần ghép
integrated_raw = ratings.merge(books[book_columns], on="ISBN", how="left", validate="many_to_one", indicator="book_match")

# Ghép tiếp người dùng theo User-ID
integrated_raw = integrated_raw.merge(users, on="User-ID", how="left", validate="many_to_one", indicator="user_match")

# Đảm bảo việc ghép không làm thay đổi số dòng đánh giá.
assert len(integrated_raw) == len(ratings)

# Xem kích thước bảng ghép và số dòng khớp/không khớp với từng bảng phụ.
print("Kích thước bảng ghép:", integrated_raw.shape)
display(integrated_raw["book_match"].value_counts())
display(integrated_raw["user_match"].value_counts())
display(integrated_raw.head())


## 3. Data Cleaning
- Bỏ dòng trùng hoàn toàn; nếu còn nhiều điểm khác nhau cho cùng cặp người dùng–sách thì dừng để xem xét, không tự chọn một điểm.
- Loại riêng 3 dòng sách có năm không phải số khỏi bảng phân tích vì có dấu hiệu lệch trường; không suy đoán để sửa. Các đánh giá liên quan vẫn ở bảng ghép, được đánh dấu không có metadata sách sạch.
- Quy ước tuổi hợp lệ 5–100 và năm xuất bản 1450–2026; đây là ngưỡng đề xuất, không phải kết luận mọi giá trị ngoài khoảng đều sai. Năm 0 coi là chưa biết. Giữ giá trị thiếu dạng số, tạo cờ thiếu; chưa điền median trước khi chia train/test để tránh rò rỉ dữ liệu.
- Điền Unknown cho tác giả, nhà xuất bản và vị trí thiếu/rỗng; chuẩn hóa khoảng trắng. ISBN luôn là chuỗi.
- Giữ điểm 0 ở bảng tổng hợp. Với tập phân loại đề xuất, chỉ lấy điểm 1–10 và đặt liked = 1 nếu điểm ≥ 8. Ý nghĩa của điểm 0 cần xác nhận theo mô tả dataset; không tự gán 0 thành nhãn không thích.


In [ ]:
# Bỏ các dòng trùng hoàn toàn và tạo bản sao để giữ nguyên ba bảng gốc.
books_clean = books.drop_duplicates().copy()
users_clean = users.drop_duplicates().copy()
ratings_clean = ratings.drop_duplicates().copy()


# Kiểm tra khóa sách duy nhất và không còn cặp người dùng–sách trùng cần xem xét.
assert books_clean["ISBN"].is_unique
assert not ratings_clean.duplicated(["User-ID", "ISBN"]).any(), "Cần xem xét đánh giá xung đột."


# Chuyển năm sang số; giá trị thiếu hoặc không chuyển được sẽ thành NaN.
parsed_year = pd.to_numeric(books_clean["Year-Of-Publication"], errors="coerce")


# Tách riêng các dòng có năm không chuyển được hoặc bị thiếu; loc lọc dòng theo điều kiện.
books_quarantine = books_clean.loc[parsed_year.isna()].copy()
books_clean = books_clean.loc[parsed_year.notna()].copy()


# Chuyển lại cột năm của bảng sách còn giữ để chuẩn hóa kiểu số.
y = pd.to_numeric(books_clean["Year-Of-Publication"], errors="coerce")
# Chỉ giữ năm 1450–2026; where thay các năm ngoài khoảng (kể cả 0) bằng giá trị thiếu.
books_clean["Year-Of-Publication"] = y.where(y.between(1450, 2026)).astype("Int64")
# Đánh giấu các năm nằm ngoài khoảng bị gán giá trị là NA
books_clean["Year_missing"] = books_clean["Year-Of-Publication"].isna()


# Chuyển tuổi sang số; giá trị không chuyển được thành giá trị thiếu.
a = pd.to_numeric(users_clean["Age"], errors="coerce")
# Theo quy ước của bài, chỉ giữ tuổi 5–100; tuổi như 244 thành <NA>, không xóa dòng người dùng.
users_clean["Age"] = a.where(a.between(5, 100)).astype("Float64")
# Đánh giấy người có tuổi ngoài khoảng hoặc bị thiếu  bị gán giá trị là NA
users_clean["Age_missing"] = users_clean["Age"].isna()


# Làm sạch các cột chữ: bỏ khoảng trắng ở hai đầu, đổi chuỗi rỗng thành thiếu rồi điền Unknown.
for frame, cols in [(books_clean, ["Book-Title", "Book-Author", "Publisher"]), (users_clean, ["Location"])]:
    for col in cols:
        frame[col] = frame[col].astype("string").str.strip().replace("", pd.NA).fillna("Unknown")
# Chuẩn hóa User-ID thành số nguyên ở cả hai bảng; errors="raise" báo lỗi nếu không chuyển được.
for frame in [users_clean, ratings_clean]:
    frame["User-ID"] = pd.to_numeric(frame["User-ID"], errors="raise").astype("int64")


# Chuyển điểm đánh giá sang số; kiểm tra mọi điểm nằm trong khoảng 0–10 và là số nguyên.
ratings_clean["Book-Rating"] = pd.to_numeric(ratings_clean["Book-Rating"], errors="raise")
assert ratings_clean["Book-Rating"].between(0, 10).all()
assert (ratings_clean["Book-Rating"] % 1 == 0).all()

# Lưu điểm bằng int8 vì khoảng 0–10 nhỏ, giúp tiết kiệm bộ nhớ.
ratings_clean["Book-Rating"] = ratings_clean["Book-Rating"].astype("int8")
# Báo số sách tách riêng, số tuổi thiếu và số năm thiếu sau khi làm sạch.
print("Sách tách riêng:", len(books_quarantine))
print("Tuổi thiếu sau xử lý:", users_clean["Age_missing"].sum())
print("Năm thiếu sau xử lý:", books_clean["Year_missing"].sum())


In [ ]:
# Ghép lại các bảng đã làm sạch; left vẫn giữ các đánh giá chưa tìm thấy thông tin sách sạch.
data_clean = ratings_clean.merge(books_clean[book_columns + ["Year_missing"]], on="ISBN", how="left", validate="many_to_one", indicator="book_match")
data_clean = data_clean.merge(users_clean, on="User-ID", how="left", validate="many_to_one", indicator="user_match")

# Kiểm tra bảng ghép giữ nguyên số đánh giá và không sinh cặp người dùng–sách trùng.
assert len(data_clean) == len(ratings_clean)
assert not data_clean.duplicated(["User-ID", "ISBN"]).any()

# Cập nhật cờ thiếu sau khi ghép, kể cả các dòng không khớp bảng sách/người dùng.
data_clean["Year_missing"] = data_clean["Year-Of-Publication"].isna()
data_clean["Age_missing"] = data_clean["Age"].isna()

# Tạo tập phân loại chỉ từ điểm 1–10; điểm 0 vẫn được giữ trong data_clean.
classification_data = data_clean.loc[data_clean["Book-Rating"].between(1, 10)].copy()


# Tạo nhãn liked:
# 1 = đánh giá cao (điểm 8–10).
# 0 = chưa đạt mức đánh giá cao (điểm 1–7).
classification_data["liked"] = (classification_data["Book-Rating"] >= 8).astype("int8")


# Xem kích thước dữ liệu và số đánh giá không có thông tin sách sạch (left_only).
print("Bảng sạch:", data_clean.shape)
print("Lượt đánh giá chưa có metadata sách sạch:", data_clean["book_match"].eq("left_only").sum())
print("Tập phân loại đề xuất:", classification_data.shape)

# Đếm số mẫu ở mỗi nhãn 0 và 1 để xem phân bố nhãn.
print(classification_data["liked"].value_counts().sort_index().to_string())

# Hiển thị 5 dòng đầu của bảng sạch để kiểm tra kết quả.
display(data_clean.head())
display(classification_data.head())


In [ ]:
# Thống kê thiếu riêng trong tập phân loại sau khi ghép dữ liệu sạch.
missing_summary = pd.DataFrame({
    "Số giá trị thiếu": classification_data.isna().sum(),
    "Tỷ lệ thiếu (%)": classification_data.isna().mean() * 100
})
print("Giá trị thiếu trong tập phân loại sau khi ghép:")
display(missing_summary.round(2))

# Phân biệt thiếu metadata do không khớp sách với thiếu từng trường dữ liệu.
missing_book_count = classification_data["book_match"].eq("left_only").sum()
missing_book_percentage = missing_book_count / len(classification_data) * 100
print(f"Lượt đánh giá không có metadata sách sạch: {missing_book_count:,} ({missing_book_percentage:.2f}%)")

# Các cờ thiếu phải đầy đủ và phản ánh đúng dữ liệu sau khi ghép.
assert classification_data["Year_missing"].equals(classification_data["Year-Of-Publication"].isna())
assert classification_data["Age_missing"].equals(classification_data["Age"].isna())


### Giá trị thiếu sau khi ghép dữ liệu

Left join giữ lại các lượt đánh giá không tìm thấy sách tương ứng, nên metadata sách có thể bị thiếu sau khi ghép dù các cột chữ trong bảng sách đã được điền `Unknown`. Bảng trên thống kê riêng số lượng và tỷ lệ thiếu trong `classification_data`.

Nhóm giữ các lượt đánh giá này và cập nhật `Year_missing`, `Age_missing` theo dữ liệu sau khi ghép. Cờ `Year_missing = True` biểu thị thiếu năm xuất bản; cột `book_match` giúp phân biệt trường hợp không khớp sách.

**Cách xử lý dự kiến khi xây dựng mô hình:**

- Với biến số được chọn làm đầu vào: điền giá trị thiếu bằng median tính trên tập train.
- Với biến phân loại được chọn làm đầu vào: điền giá trị thiếu bằng `Unknown`.
- Áp dụng cùng cách xử lý cho tập validation/test, sử dụng các tham số đã học từ tập train.

Chưa điền median ở bước này để tránh rò rỉ dữ liệu. Các cờ thiếu không bị thiếu giá trị vì được tính lại bằng `isna()`; số giá trị thiếu của các biến gốc vẫn được thể hiện trong bảng thống kê.


### Chuyển sang EDA và xây dựng mô hình
Các bảng sạch được dùng ở phần 4–5 dưới đây. Đây là bài toán dự đoán một lượt đánh giá có điểm ≥ 8 hay không, trong nhóm đánh giá có điểm 1–10. Không suy rộng kết quả thành hành vi của tất cả người dùng hoặc khả năng họ sẽ mua/đọc sách. Điểm 0 được giữ ở bảng tổng hợp nhưng chưa dùng làm nhãn phân loại vì ý nghĩa cần được xác nhận.

## 4. Exploratory Data Analysis (EDA)

### 4.1. PHÂN TÍCH ĐỘ TUỔI NGƯỜI DÙNG

In [ ]:
# Chuyển tuổi sang kiểu số để thống kê.
# Tuổi thiếu hoặc không chuyển được thành số sẽ trở thành NaN.
user_age = pd.to_numeric(users_clean["Age"],errors="coerce")
# Thống kê tuổi của người dùng.
print("\nThống kê tuổi người dùng:")
print(user_age.describe())
# Đếm số người dùng thiếu tuổi và tính tỷ lệ phần trăm.
missing_age_count = user_age.isna().sum()
missing_age_percentage = user_age.isna().mean() * 100
print("Số người dùng thiếu tuổi:", missing_age_count)
print(f"Tỷ lệ người dùng thiếu tuổi: {missing_age_percentage:.2f}%")
# Vẽ biểu đồ phân phối tuổi, bỏ qua các giá trị thiếu.
plt.figure(figsize=(10, 5))
user_age.dropna().plot(kind="hist",bins=20,edgecolor="black")
plt.title("Phân phối độ tuổi người dùng")
plt.xlabel("Tuổi")
plt.ylabel("Số lượng người dùng")
plt.tight_layout()
plt.show()


### 4.2. PHÂN TÍCH PHÂN PHỐI ĐIỂM ĐÁNH GIÁ

In [ ]:

# Đếm số lượt đánh giá thuộc mỗi nhóm:
# liked = 1: đánh giá cao (điểm 8–10)
# liked = 0: chưa đạt mức đánh giá cao (điểm 1–7)
label_counts = (classification_data["liked"].value_counts().reindex([1, 0], fill_value=0))
# Đổi nhãn số thành tên nhóm để biểu đồ dễ đọc hơn.
label_counts.index = ["Đánh giá cao (8–10)", "Chưa đạt mức đánh giá cao (1–7)"]
# Tính tỷ lệ phần trăm của mỗi nhóm trên tổng số lượt đánh giá.
label_percentages = (label_counts / label_counts.sum() * 100)
# Tổng hợp số lượng và tỷ lệ thành bảng thống kê.
label_summary = pd.DataFrame({"Số lượt đánh giá": label_counts,"Tỷ lệ (%)": label_percentages})


# Hiển thị bảng thống kê để kiểm tra kết quả.
print("Phân bố hai nhóm đánh giá:")
display(label_summary.round(2))


# Vẽ biểu đồ cột thể hiện số lượng của hai nhóm.
plt.figure(figsize=(10, 6))
bars = plt.bar(label_counts.index,label_counts.values,edgecolor="black")


# Ghi số lượt đánh giá và tỷ lệ phần trăm lên mỗi cột.
for bar, count, percentage in zip(bars,label_counts.values,label_percentages.values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{count:,}\n({percentage:.2f}%)",
        ha="center",
        va="bottom",
        fontsize=11
    )
# Đặt tiêu đề, tên trục và căn chỉnh biểu đồ.
plt.title("Phân bố nhãn đánh giá sách")
plt.xlabel("Nhóm đánh giá")
plt.ylabel("Số lượt đánh giá")
plt.ylim(0, label_counts.max() * 1.15)
plt.tight_layout()
plt.show()

In [ ]:
# Tách các lượt đánh giá có điểm tường minh từ 1 đến 10.
# Giữ nguyên ratings_clean để không làm thay đổi dữ liệu gốc đã làm sạch.
explicit_ratings = ratings_clean.loc[ratings_clean["Book-Rating"].between(1, 10)].copy()
# Đếm số lượt đánh giá theo từng điểm từ 1 đến 10.
explicit_rating_counts = (explicit_ratings["Book-Rating"].value_counts().sort_index())
# Tính tỷ lệ phần trăm của từng điểm trong nhóm đánh giá tường minh.
explicit_rating_percentages = (explicit_ratings["Book-Rating"].value_counts(normalize=True).sort_index() * 100)
# Hiển thị số lượng và tỷ lệ để so sánh các mức điểm.
print("Số lượt đánh giá theo điểm:")
print(explicit_rating_counts)
print("\nTỷ lệ phần trăm theo điểm:")
print(explicit_rating_percentages.round(2))
# Vẽ biểu đồ phân phối điểm đánh giá tường minh.
plt.figure(figsize=(10, 5))
explicit_rating_counts.plot(kind="bar", edgecolor="black")
plt.title("Phân phối điểm đánh giá tường minh (1–10)")
plt.xlabel("Điểm đánh giá")
plt.ylabel("Số lượt đánh giá")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 4.3. NGƯỜI DÙNG CÓ NHIỀU LƯỢT ĐÁNH GIÁ CÓ ĐIỂM NHẤT (1–10)

Đếm các lượt có điểm 1–10; không tính điểm 0.


In [ ]:
# Chỉ giữ các lượt đánh giá có điểm từ 1 đến 10.
# Book-Rating = 0 không được tính vì không phải điểm đánh giá rõ ràng.
explicit_ratings = ratings_clean.loc[ratings_clean["Book-Rating"].between(1, 10)].copy()
# Đếm số lượt đánh giá có điểm của từng người dùng.
user_rating_counts = (explicit_ratings.groupby("User-ID").size().reset_index(name="Number_of_Ratings"))
# Sắp xếp số lượt đánh giá từ cao xuống thấp và lấy 10 người dùng đầu tiên.
top_10_users = (user_rating_counts.sort_values("Number_of_Ratings", ascending=False).head(10).reset_index(drop=True))
# Đánh số thứ hạng từ 1 đến 10 để dễ theo dõi.
top_10_users.index = top_10_users.index + 1
top_10_users.index.name = "Rank"
# Hiển thị bảng kết quả.
print("TOP 10 NGƯỜI DÙNG CÓ NHIỀU LƯỢT ĐÁNH GIÁ CÓ ĐIỂM NHẤT (1–10):")
display(top_10_users)

### 4.4. SÁCH CÓ NHIỀU LƯỢT ĐÁNH GIÁ NHẤT (0–10)

Đếm toàn bộ lượt trong bảng Ratings, bao gồm điểm 0. Số đếm này phản ánh mức độ xuất hiện trong dữ liệu, khác với số lượt có điểm 1–10 ở mục 4.3.


In [ ]:
# Đếm số lượt đánh giá cho mỗi ISBN.
# Kết quả thể hiện mức độ xuất hiện của sách trong bảng Ratings.
book_rating_counts = (ratings_clean.groupby("ISBN").size().sort_values(ascending=False))
# Lấy 10 ISBN có nhiều lượt đánh giá nhất.
top_10_isbns = book_rating_counts.head(10)
# Tạo bảng thông tin sách để bổ sung tên sách cho kết quả.
# Chỉ lấy ISBN và tên sách để tránh ghép các cột không cần thiết.
book_titles = books_clean[["ISBN", "Book-Title"]].drop_duplicates(subset=["ISBN"])
# Ghép số lượt đánh giá với tên sách tương ứng.
top_10_books = (top_10_isbns.rename("Number_of_Ratings").reset_index().merge(book_titles, on="ISBN", how="left"))
# Hiển thị 10 sách có nhiều lượt đánh giá nhất.
print("\nTop 10 sách có nhiều lượt đánh giá nhất (0–10, bao gồm điểm 0):")
display(top_10_books)


### 4.5. PHÂN TÍCH MỨC ĐIỂM ĐÁNH GIÁ CỦA SÁCH

In [ ]:
# Chỉ lấy những lượt đánh giá từ 1 đến 10.
# Điểm 0 không được đưa vào phép tính điểm trung bình.
explicit_rating_data = ratings_clean.loc[ratings_clean["Book-Rating"].between(1, 10)].copy()
# Tính điểm trung bình và số lượt đánh giá cho từng ISBN.
book_rating_summary = (explicit_rating_data.groupby("ISBN")["Book-Rating"].agg(["mean", "count"]).reset_index())
# Đổi tên cột để kết quả dễ đọc hơn.
book_rating_summary.columns = ["ISBN","Average_Rating","Number_of_Ratings"]
# Chỉ xét sách có ít nhất 10 lượt đánh giá để hạn chế
# việc sách có quá ít lượt đánh giá đứng đầu do ngẫu nhiên.
popular_books = book_rating_summary.loc[book_rating_summary["Number_of_Ratings"] >= 10].copy()
# Sắp xếp theo điểm trung bình giảm dần và lấy 10 sách đầu tiên.
top_10_best_books = popular_books.sort_values(by=["Average_Rating", "Number_of_Ratings"],ascending=[False, False]).head(10)
# Bổ sung tên sách để dễ diễn giải kết quả.
top_10_best_books = top_10_best_books.merge(book_titles,on="ISBN",how="left")
# Sắp xếp theo điểm trung bình giảm dần và lấy 10 sách đầu tiên.
top_10_worst_books = popular_books.sort_values(by=["Average_Rating", "Number_of_Ratings"],ascending=[True, False]).head(10)
# Bổ sung tên sách để dễ diễn giải kết quả.
top_10_worst_books = top_10_worst_books.merge(book_titles,on="ISBN",how="left")
# Hiển thị 10 sách có điểm trung bình cao nhất trong nhóm đủ lượt đánh giá.
print("\nTop 10 sách có điểm trung bình cao nhất (ít nhất 10 lượt đánh giá):")
display(top_10_best_books)
print("---------------------")
print("\nTop 10 sách có điểm trung bình thấp nhất (ít nhất 10 lượt đánh giá):")
display(top_10_worst_books)


### 4.6. PHÂN TÍCH MỐI QUAN HỆ GIỮA ĐỘ TUỔI VÀ NHÃN ĐÁNH GIÁ

In [ ]:
# Tạo bản sao để không làm thay đổi classification_data hiện có.
age_label_data = classification_data.copy()
# Chuyển Age về dạng số; giá trị không hợp lệ sẽ thành NaN.
age_label_data["Age"] = pd.to_numeric(age_label_data["Age"],errors="coerce")
# Chỉ giữ độ tuổi hợp lệ trong khoảng 5–100.
age_label_data.loc[~age_label_data["Age"].between(5, 100),"Age"] = np.nan
# Thống kê số lượng, trung bình và trung vị độ tuổi theo nhãn.
age_label_summary = (age_label_data.groupby("liked")["Age"].agg(["count", "mean", "median", "std"]))
# Đổi tên nhãn để phân biệt rõ ý nghĩa.
age_label_summary.index = ["Nhãn 0 (điểm 1–7)","Nhãn 1 (điểm 8–10)"]
print("Thống kê độ tuổi theo nhãn:")
display(age_label_summary.round(2))
# Tạo các nhóm tuổi để so sánh tỷ lệ nhãn giữa các nhóm.
age_label_data["Age_Group"] = pd.cut(age_label_data["Age"],bins=[4, 18, 30, 45, 60, 100],labels=["5–18", "19–30", "31–45", "46–60", "61–100"])
# Tính tỷ lệ nhãn 0 và 1 trong từng nhóm tuổi.
age_label_distribution = pd.crosstab(age_label_data["Age_Group"],age_label_data["liked"],normalize="index") * 100
# Đổi tên cột cho dễ đọc.
age_label_distribution = age_label_distribution.rename(columns={0: "Nhãn 0 (điểm 1–7)",1: "Nhãn 1 (điểm 8–10)"})
print("Tỷ lệ nhãn trong từng nhóm tuổi (%):")
display(age_label_distribution.round(2))
# Tạo biểu đồ cột; mỗi nhóm tuổi có hai cột tương ứng với hai nhãn(# Đặt hai cột cạnh nhau thay vì chồng lên nhau).
ax = age_label_distribution.plot(kind="bar",stacked=False,             figsize=(11, 6),width=0.8,edgecolor="black")
# Thêm tiêu đề và tên các trục.
ax.set_title("So sánh tỷ lệ hai nhãn đánh giá theo nhóm tuổi",fontsize=14,fontweight="bold",pad=15)
ax.set_xlabel("Nhóm tuổi", fontsize=12)
ax.set_ylabel("Tỷ lệ đánh giá (%)", fontsize=12)
# Giữ nguyên thứ tự và cách hiển thị tên các nhóm tuổi.
ax.tick_params(axis="x", rotation=0)
# Đặt giới hạn trục tung từ 0 đến 100% để dễ so sánh.
ax.set_ylim(0, 100)
# Hiển thị đường lưới ngang để dễ đọc giá trị.
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Tỷ lệ lượt đánh giá cao (điểm 8–10) trong các lượt có điểm 1–10 theo nhóm tuổi
# Chỉ sử dụng các bản ghi có tuổi hợp lệ.
age_label_data = classification_data.dropna(subset=["Age", "liked"]).copy()
# Chia tuổi thành các nhóm, mỗi nhóm cách nhau 10 năm.
age_bins = [5, 15, 25, 35, 45, 55, 65, 75, 85, 95, 101]
age_labels = ["5–14", "15–24", "25–34", "35–44", "45–54","55–64", "65–74", "75–84", "85–94", "95–100"]
age_label_data["Age_Group"] = pd.cut(age_label_data["Age"],bins=age_bins,labels=age_labels,right=False,include_lowest=True)
# Tính số lượt đánh giá và tỷ lệ nhãn 1 theo nhóm tuổi.
age_summary = (age_label_data.groupby("Age_Group", observed=False).agg(Number_of_Ratings=("liked", "size"),Positive_Rate=("liked", "mean")).reset_index())
# Chuyển tỷ lệ nhãn 1 thành phần trăm.
age_summary["Positive_Rate"] *= 100
# Vẽ biểu đồ tỷ lệ nhãn 1 theo nhóm tuổi.
plt.figure(figsize=(10, 5))
plt.bar(age_summary["Age_Group"].astype(str),age_summary["Positive_Rate"])
plt.title("Tỷ lệ lượt đánh giá cao theo nhóm tuổi (chỉ xét điểm 1–10)")
plt.xlabel("Nhóm tuổi")
plt.ylabel("Tỷ lệ lượt đánh giá cao (%)")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
# Lập bảng chéo để kiểm định mối liên hệ giữa nhóm tuổi và nhãn.
contingency_table = pd.crosstab(age_label_data["Age_Group"],age_label_data["liked"])
# Chi-square tham khảo ở cấp lượt đánh giá; các lượt của cùng người dùng có thể phụ thuộc nhau.
chi2, p_value, dof, expected = chi2_contingency(contingency_table)
print("Tỷ lệ nhãn 1 theo nhóm tuổi:")
display(age_summary.round(2))
print(f"Chi-square statistic: {chi2:.4f}")
print(f"P-value Chi-square thông thường (chỉ tham khảo): {p_value:.6g}")
# Không kết luận dựa trên ngưỡng p-value khi chưa bảo đảm tính độc lập.
print("Diễn giải: Bảng và biểu đồ mô tả tỷ lệ lượt đánh giá cao theo nhóm tuổi trong mẫu quan sát.")
print("Hạn chế: Một người dùng có thể đóng góp nhiều lượt đánh giá; giả định độc lập của Chi-square có thể bị vi phạm.")
print("Vì vậy, p-value trên chỉ mang tính tham khảo, chưa đủ để khẳng định mối liên hệ trong tổng thể.")


**Phạm vi diễn giải mục 4.6:** Mỗi dòng là một lượt đánh giá có điểm 1–10, không phải một người dùng độc lập. Biểu đồ thể hiện tỷ lệ lượt đạt điểm 8–10 trong từng nhóm tuổi. Người dùng đánh giá nhiều đóng góp nhiều dòng; các lượt còn có thể cùng liên quan đến một sách. Giữ Chi-square như kết quả tham khảo, không dùng p-value này để khẳng định mối liên hệ trong tổng thể hoặc quan hệ nhân quả. Nếu cần suy luận thống kê, cần thiết kế phân tích xử lý sự phụ thuộc giữa các quan sát, chẳng hạn tổng hợp theo người dùng và xem xét ảnh hưởng của sách. Việc tổng hợp làm thay đổi đơn vị phân tích và phải được giải thích rõ.


### 4.7. PHÂN TÍCH MỐI QUAN HỆ GIỮA ĐỘ TUỔI VÀ SỐ LƯỢT ĐÁNH GIÁ SÁCH

In [ ]:
# Tạo biểu đồ Log Scale
# Lấy các bản ghi có tuổi hợp lệ.
age_rating_data = classification_data.dropna(subset=["Age", "User-ID"]).copy()
# Đếm số lượt đánh giá của mỗi người dùng.
user_rating_counts = (age_rating_data.groupby(["User-ID", "Age"]).size().reset_index(name="Number_of_Ratings"))
# Vẽ biểu đồ Scatter.
plt.figure(figsize=(10, 6))
plt.scatter(user_rating_counts["Age"],user_rating_counts["Number_of_Ratings"],alpha=0.35, s=18)
# Dùng thang logarit để thể hiện rõ các giá trị nhỏ và lớn.
plt.yscale("log")
plt.title("User Age vs Number of Book Ratings (Log Scale)")
plt.xlabel("User Age")
plt.ylabel("Number of Ratings (Log Scale)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Tạo biểu đồ Scatter.
plt.figure(figsize=(10, 6))
plt.scatter(user_rating_counts["Age"],user_rating_counts["Number_of_Ratings"],alpha=0.3,s=15)
# Đặt tên biểu đồ và các trục.
plt.title("Relationship Between User Age and Number of Book Ratings")
plt.xlabel("User Age")
plt.ylabel("Number of Ratings")
# Hiển thị lưới để dễ quan sát.
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.8. MỨC ĐỘ HOẠT ĐỘNG CỦA NGƯỜI DÙNG VÀ TỶ LỆ ĐÁNH GIÁ POSITIVE 

In [ ]:
# Tổng hợp số lượt đánh giá và tỷ lệ nhãn 1 của từng người dùng.
user_activity = (classification_data.groupby("User-ID").agg(Number_of_Ratings=("liked", "size"),Positive_Rate=("liked", "mean")).reset_index())
# Tính ngưỡng phân vị 25% và 75% của số lượt đánh giá.
q25 = user_activity["Number_of_Ratings"].quantile(0.25)
q75 = user_activity["Number_of_Ratings"].quantile(0.75)
# Chỉ giữ nhóm hoạt động thấp và nhóm hoạt động cao.
low_activity = user_activity[user_activity["Number_of_Ratings"] <= q25].copy()
high_activity = user_activity[user_activity["Number_of_Ratings"] >= q75
].copy()
# Tổng hợp số người dùng và tỷ lệ nhãn 1 trung bình trong mỗi nhóm.
activity_summary = pd.DataFrame({"Group": ["Low Activity", "High Activity"],"Number_of_Users": [len(low_activity),len(high_activity)],
    "Median_Ratings_Per_User": [
        low_activity["Number_of_Ratings"].median(),
        high_activity["Number_of_Ratings"].median()
    ],
    "Average_Positive_Rate": [
        low_activity["Positive_Rate"].mean() * 100,
        high_activity["Positive_Rate"].mean() * 100
    ]
})
# Vẽ biểu đồ so sánh tỷ lệ nhãn 1 trung bình của mỗi người dùng.
plt.figure(figsize=(8, 5))
plt.bar(activity_summary["Group"],activity_summary["Average_Positive_Rate"])
plt.title("Average Positive Rating Rate by User Activity")
plt.xlabel("User Activity Group")
plt.ylabel("Average Positive Rate (%)")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
# Kiểm định Mann–Whitney U trên tỷ lệ nhãn 1 của từng người dùng.
# Mỗi người dùng chỉ đóng góp một giá trị tỷ lệ vào kiểm định.
u_statistic, p_value = mannwhitneyu(low_activity["Positive_Rate"],high_activity["Positive_Rate"],alternative="two-sided")
print("Bảng so sánh mức độ hoạt động:")
display(activity_summary.round(2))
print(f"Mann–Whitney U statistic: {u_statistic:.4f}")
print(f"P-value: {p_value:.6g}")
# So sánh tỷ lệ nhãn 1 trung bình giữa hai nhóm.
low_rate = low_activity["Positive_Rate"].mean() * 100
high_rate = high_activity["Positive_Rate"].mean() * 100
print(f"Chênh lệch tỷ lệ nhãn 1: {high_rate - low_rate:.2f} điểm phần trăm")
if p_value < 0.05:
    print("Kết luận: Có bằng chứng thống kê cho thấy phân phối tỷ lệ ""nhãn 1 khác nhau giữa nhóm hoạt động thấp và cao.")
else:
    print("Kết luận: Chưa có đủ bằng chứng thống kê về sự khác biệt ""tỷ lệ nhãn 1 giữa hai nhóm.")

### 4.9. EDA Summary

Kết quả khám phá dữ liệu (EDA) cho thấy bốn phát hiện đáng chú ý:

- **Phân phối nhãn:** Nhãn 1 (điểm đánh giá 8–10) chiếm 57,62%, cao hơn nhãn 0 (điểm đánh giá 1–7) với 42,38%. Mặc dù có sự chênh lệch giữa hai nhãn, mức độ mất cân bằng chưa quá lớn. Tuy nhiên, cần xem xét phân phối nhãn khi đánh giá hiệu quả mô hình phân loại.

- **Hành vi người dùng:** Số lượt đánh giá phân bố không đồng đều, với một số người dùng hoạt động vượt trội so với phần còn lại. Phân tích sâu hơn cho thấy nhóm người dùng có ít lượt đánh giá có tỷ lệ nhãn 1 trung bình là 56,27%, cao hơn nhóm hoạt động cao với 53,00%, chênh lệch khoảng 3,28 điểm phần trăm. Kiểm định Mann–Whitney U cho p-value = 3,21475 × 10⁻¹²⁵, cung cấp bằng chứng thống kê về sự khác biệt trong phân phối tỷ lệ nhãn 1 giữa hai nhóm. Kết quả gợi ý mức độ hoạt động có liên quan đến hành vi đánh giá, nhưng không chứng minh số lượt đánh giá nhiều khiến người dùng cho điểm thấp hơn.

- **Mức độ phổ biến của sách:** Các bảng top ở mục 4.4–4.5 mô tả hai tiêu chí khác nhau: số lượt đánh giá (bao gồm điểm 0) và điểm trung bình (chỉ tính điểm 1–10, với ít nhất 10 lượt có điểm). Việc đối chiếu các bảng này mới mang tính mô tả; chưa đủ để kết luận về mối quan hệ giữa độ phổ biến và điểm trung bình trên toàn bộ sách.

- **Vai trò của độ tuổi:** Trong mẫu quan sát, tỷ lệ lượt đánh giá cao khác nhau giữa các nhóm tuổi. Nhóm 45–54 tuổi đạt 64,93%; nhóm 85–94 tuổi đạt 34,67% nhưng chỉ có 75 lượt đánh giá. Các tỷ lệ được tính theo lượt đánh giá, nên người dùng có nhiều lượt đóng góp trọng số lớn hơn. Chi-square thông thường cho p-value = 2,61537 × 10⁻¹⁴⁶, nhưng một người dùng có thể đóng góp nhiều lượt, khiến giả định độc lập có thể bị vi phạm. Vì vậy, p-value chỉ được báo cáo để tham khảo, chưa đủ để khẳng định mối liên hệ trong tổng thể. Khoảng 40,17% người dùng thiếu tuổi sau làm sạch; đây là tỷ lệ ở cấp người dùng, khác với tỷ lệ thiếu tuổi trong tập lượt đánh giá phân loại.


**Kết luận:** EDA mô tả sự khác biệt về hành vi đánh giá trong mẫu theo người dùng và nhóm tuổi. Các bảng top sách giúp xem xét riêng số lượt đánh giá và điểm trung bình; chưa xác lập mối quan hệ giữa hai đại lượng trên toàn bộ sách. Đây là cơ sở để lựa chọn và kiểm tra các đặc trưng tiềm năng khi xây dựng mô hình phân loại. Tuy nhiên, mối liên hệ có ý nghĩa thống kê không đồng nghĩa với khả năng dự đoán tốt, vì vậy cần đánh giá vai trò thực tế của từng đặc trưng trên tập validation trước khi đánh giá cuối cùng trên test. Quá trình xây dựng mô hình cũng cần xem xét dữ liệu tuổi bị thiếu, sự khác biệt về mức độ hoạt động giữa người dùng và khả năng tổng quát hóa của mô hình. Đặc biệt, không sử dụng `Book-Rating` làm đặc trưng đầu vào vì nhãn `liked` được tạo trực tiếp từ cột này, dẫn đến rò rỉ dữ liệu (*data leakage*).

### 4.10. Target và các đặc trưng dự kiến

Đơn vị dự đoán là một lượt đánh giá của một người dùng đối với một sách. Tập phân loại chỉ gồm các lượt có `Book-Rating` từ 1–10. Bảng dưới xác định target và các đầu vào dự kiến để thử nghiệm ở phần 5; đây chưa phải kết luận đặc trưng nào dự đoán tốt nhất.

| Vai trò | Biến | Lý do chọn / ý nghĩa | Cách xử lý dự kiến |
|---|---|---|---|
| Target | `liked` | 1: điểm 8–10; 0: điểm 1–7. Ngưỡng 8 là quy ước của nhóm. | Giữ dạng nhị phân; không đưa vào X. |
| Đầu vào ban đầu | `Age` | EDA cho thấy tỷ lệ đánh giá cao khác nhau giữa các nhóm tuổi; cần kiểm tra khả năng dự đoán thực tế. | Điền median từ train; chuẩn hóa nếu thuật toán cần. |
| Đầu vào ban đầu | `Age_missing` | Phân biệt tuổi quan sát được với tuổi được điền. | Chuyển cờ boolean thành 0/1. |
| Đầu vào ban đầu | `Year-Of-Publication` | Thử kiểm tra liệu năm xuất bản có giúp phân biệt hai nhãn; EDA hiện chưa chứng minh mối liên hệ này. | Điền median từ train; chuẩn hóa nếu thuật toán cần. |
| Đầu vào ban đầu | `Year_missing` | Đánh dấu năm thiếu hoặc không hợp lệ, kể cả trường hợp không khớp sách. | Dùng cờ đã cập nhật sau merge, chuyển thành 0/1. |
| Đầu vào mở rộng | `Book-Author`, `Publisher` | Thử khai thác khác biệt giữa tác giả/nhà xuất bản; chưa có bằng chứng EDA trực tiếp về khả năng dự đoán. | Điền `Unknown`; học nhóm hiếm và bộ mã hóa trên train; xử lý được nhóm chưa gặp ở validation/test. |
| Đầu vào mở rộng | Quốc gia trích từ `Location` | Thử biểu diễn vị trí ở mức khái quát, giảm số lượng giá trị so với chuỗi vị trí đầy đủ. | Tạo biến mới bằng quy tắc tách và chuẩn hóa được kiểm tra; giá trị không xác định thành `Unknown`; mã hóa trên train. Biến này chưa được tạo ở phần EDA. |
| Đầu vào mở rộng | Số lượt đánh giá theo người dùng/sách | EDA cho thấy mức hoạt động và mức phổ biến không đồng đều; có thể cung cấp tín hiệu dự đoán. | Tạo biến mới chỉ từ train; khi tạo giá trị cho chính train, loại lượt hiện tại khỏi số đếm. Với validation/test, tra bảng thống kê từ train; ID chưa gặp gán 0. |
| Đầu vào nâng cao, tùy chọn | Điểm trung bình hoặc tỷ lệ nhãn 1 theo người dùng/sách | Thử biểu diễn xu hướng đánh giá; có nguy cơ rò rỉ vì sử dụng điểm/nhãn. | Không dùng các bảng tổng hợp toàn bộ dữ liệu ở EDA làm đầu vào. Tạo đặc trưng train bằng out-of-fold: mỗi dòng nhận thống kê từ các fold khác; validation/test chỉ dùng thống kê từ train. Nhóm chưa gặp dùng giá trị chung của train. |

**Các cột không dùng trực tiếp làm đầu vào:**

- `Book-Rating`: loại khỏi X vì `liked` được tạo trực tiếp từ điểm này, sẽ gây rò rỉ target.
- `User-ID`, `ISBN`: giữ làm khóa ghép, chia dữ liệu và tính thống kê; không coi các mã định danh là biến số liên tục.
- `Book-Title` và chuỗi `Location` đầy đủ: chưa đưa vào mô hình ban đầu vì cần bước xử lý văn bản hoặc chuẩn hóa riêng.
- `book_match`: dùng để kiểm tra chất lượng ghép; chưa chọn làm đầu vào ban đầu. `user_match` không phân biệt được các mẫu trong dữ liệu hiện tại vì tất cả đều khớp người dùng.
- Các cột URL ảnh: chưa sử dụng vì phạm vi hiện tại không xử lý ảnh.

**Nguyên tắc thực hiện ở phần 5:**

1. Chia train/validation/test trước khi học median, bộ chuẩn hóa, nhóm giá trị hiếm, bộ mã hóa hoặc các thống kê theo người dùng/sách. Chọn cách chia phù hợp với mục tiêu dự đoán cho người dùng/sách đã biết hay chưa gặp.
2. Bắt đầu bằng nhóm đầu vào ban đầu; thử bổ sung các nhóm mở rộng và so sánh trên validation. Giữ test để đánh giá cuối cùng.
3. Không xem ý nghĩa thống kê trong EDA là bằng chứng chắc chắn về khả năng dự đoán. Chọn đặc trưng dựa trên kết quả validation và khả năng áp dụng thực tế.

Phần này chỉ chốt kế hoạch đặc trưng sau EDA; chưa huấn luyện mô hình hoặc tạo các đặc trưng tổng hợp cho mô hình.


## 5. Implementing Machine Learning Algorithms